In [112]:
from datetime import datetime

import ConnectionConfig as cc

cc.setupEnvironment()
debugging_mode = True

Environment variables are set...


In [113]:
#config
cc.setupEnvironment()
print(cc.config.sections())
spark = cc.startLocalCluster("FACT_TREASURE_FOUND", 4)
spark.getActiveSession()

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [114]:

#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [115]:
from pyspark import Row
import uuid

# Haal bestaande hunter_id en treasure_id op
existing_hunter = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT id FROM user_table LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

existing_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT id FROM treasure LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

hunter_id = existing_hunter.first()["id"]
treasure_id = existing_treasure.first()["id"]

new_record = spark.createDataFrame([
    Row(
        id=uuid.uuid4().bytes,
        description='Incremental load test - new treasure found!',
        log_time=datetime.now(),
        log_type=2,
        session_start=datetime.now(),
        hunter_id=hunter_id,
        treasure_id=treasure_id
    )
])

new_record.write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .mode("append") \
    .save()

print("Record toegevoegd met bestaande foreign keys")

Record toegevoegd met bestaande foreign keys


In [116]:
newest_log = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT * FROM treasure_log ORDER BY log_time DESC LIMIT 1) AS t") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

# dit haalt de meest recente log op
print("nieuwste log")
newest_log.show(truncate=False)

nieuwste log
+-------------------------------------------------+-------------------------------------------+--------------------------+--------+--------------------------+-------------------------------------------------+-------------------------------------------------+
|id                                               |description                                |log_time                  |log_type|session_start             |hunter_id                                        |treasure_id                                      |
+-------------------------------------------------+-------------------------------------------+--------------------------+--------+--------------------------+-------------------------------------------------+-------------------------------------------------+
|[DC E2 CE 10 AB 2B 41 17 8F 4C C1 41 14 4F 40 58]|Incremental load test - new treasure found!|2025-11-02 21:02:38.130259|2       |2025-11-02 21:02:38.130262|[A0 05 2A AC B9 70 42 AD B9 77 07 FF E4 80 AF 64]|[0